##  ulta top10 순위 추출코드

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time
import random
import json
from datetime import datetime

def fetch_ulta_skincare_top10():
    # 1. 드라이버 설정 (디버깅을 위해 Headless False)
    options = uc.ChromeOptions()
    options.add_argument('--headless') 
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-blink-features=AutomationControlled')
    options.add_argument('--incognito')

    user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36"
    options.add_argument(f'user-agent={user_agent}')

    driver = None
    try:
        driver = uc.Chrome(options=options)
        
        # Ulta 스킨케어 베스트셀러 정렬 페이지
        target_url = "https://www.ulta.com/shop/skin-care/all?sort=best_sellers"

        print(f"📡 Ulta 접속 시도 중...")
        driver.get(target_url)
        
        # Ulta는 페이지 로딩과 봇 체크가 엄격하므로 충분히 대기합니다.
        time.sleep(random.uniform(7, 10))

        # 2. 데이터 파싱 시작
        print("📦 데이터 추출 시작 (광고 상품 필터링 중)...")
        soup = BeautifulSoup(driver.page_source, "html.parser")
        
        # 상품 카드 리스트 아이템 선택
        items = soup.select('li.ProductListingResults__productCard')
        
        final_rankings = []
        for item in items:
            if len(final_rankings) >= 10:
                break
                
            try:
                # [중요] 광고(Sponsored) 상품 제외 체크
                # 하단 버튼 근처에 'Sponsored'라는 문구가 있는지 확인
                sponsored_tag = item.select_one(".pal-c-ProductCardFooter__sponsored")
                if sponsored_tag and "Sponsored" in sponsored_tag.get_text():
                    continue # 광고 상품이면 스킵

                # A. 브랜드명
                brand_tag = item.select_one(".pal-c-ProductCardBody--brandName p")
                brand = brand_tag.get_text(strip=True) if brand_tag else "N/A"

                # B. 상품명 및 상세 URL
                title_link = item.select_one("a.pal-c-Link")
                if title_link:
                    title = title_link.get_text(strip=True)
                    product_url = title_link.get('href')
                    if not product_url.startswith('http'):
                        product_url = "https://www.ulta.com" + product_url
                else:
                    continue

                # C. 가격 (세일 가격이 있을 경우와 일반 가격 구분)
                price_tag = item.select_one(".pal-c-Price span[aria-hidden='true']") or \
                            item.select_one(".pal-c-Price__priceContainer span")
                price = price_tag.get_text(strip=True) if price_tag else "N/A"

                # D. 평점 및 리뷰 수 (sr-only 텍스트에서 추출)
                # 구조: <span class="sr-only">4.5 out of 5 stars ; 3150 reviews</span>
                rating_info_tag = item.select_one(".pal-c-Ratings span.sr-only")
                rating = "0.0"
                reviews = "0"
                if rating_info_tag:
                    raw_text = rating_info_tag.get_text(strip=True) # "4.5 out of 5 stars ; 3150 reviews"
                    rating = raw_text.split(' ')[0]
                    # 세미콜론(;) 뒤의 숫자만 추출
                    if ';' in raw_text:
                        reviews = raw_text.split(';')[-1].replace('reviews', '').replace(',', '').strip()

                final_rankings.append({
                    "rank": len(final_rankings) + 1,
                    "brand": brand,
                    "title": title,
                    "rating": float(rating) if rating.replace('.','').isdigit() else 0.0,
                    "reviews": int(reviews) if reviews.isdigit() else 0,
                    "price": price,
                    "url": product_url,
                    "platform": "Ulta",
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                })
                print(f"✅ {len(final_rankings)}위: [{brand}] {title[:20]}...")

            except Exception as e:
                print(f"⚠️ 개별 상품 파싱 중 에러: {e}")
                continue

        # 3. JSON 저장
        filename = f"ulta_skincare_top10_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(final_rankings, f, ensure_ascii=False, indent=4)
        
        print(f"\n✨ 수집 완료! 광고 제외 순수 Top 10 저장됨: {filename}")
        return final_rankings

    except Exception as e:
        print(f"❌ 최종 에러 발생: {e}")
    finally:
        if driver:
            # 창이 닫히는 것을 방지하려면 아래 줄 주석 처리
            # time.sleep(10)
            driver.quit()

if __name__ == "__main__":
    fetch_ulta_skincare_top10()

📡 Ulta 접속 시도 중...
📦 데이터 추출 시작 (광고 상품 필터링 중)...
✅ 1위: [IT Cosmetics] IT Cosmetics Do It A...
✅ 2위: [medicube] medicube Zero Pore P...
✅ 3위: [The Ordinary] The Ordinary Glycoli...
✅ 4위: [Clinique] Clinique Moisture Su...
✅ 5위: [MAËLYS] MAËLYS GET-DREAMY Ov...
✅ 6위: [The Ordinary] The Ordinary Hyaluro...
✅ 7위: [OLEHENRIKSEN] OLEHENRIKSEN Pout Pr...
✅ 8위: [Clinique] Clinique Even Better...
✅ 9위: [TATCHA] TATCHA The Dewy Skin...
✅ 10위: [La Roche-Posay] La Roche-Posay Toler...

✨ 수집 완료! 광고 제외 순수 Top 10 저장됨: ulta_skincare_top10_20260317_140933.json


## ulta top10 순위 추출 및 리뷰데이터 추출통합코드

In [ ]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import json
import time
import random
from datetime import datetime

# --- 설정 및 파일명 (.jsonl로 변경) ---
API_KEY = "daa0f241-c242-4483-afb7-4449942d1a2b"
REVIEW_SAVE_FILE = "ulta_reviews_master.jsonl"  # 한 줄에 리뷰 1개씩 저장
RANK_SAVE_FILE = "ulta_rankings_current.jsonl"   # 한 줄에 상품 1개씩 저장

def get_product_reviews_refined(product_id, product_name):
    """PowerReviews API를 사용하여 특정 상품의 모든 리뷰를 수집하고 정제함"""
    raw_reviews = []
    page_size = 25 
    paging_from = 0
    
    print(f"\n🚀 [{product_name}] 전체 리뷰 수집 및 정제 시작...")

    while True:
        url = f"https://display.powerreviews.com/m/6406/l/en_US/product/{product_id}/reviews"
        params = {
            "paging.from": paging_from,
            "paging.size": page_size,
            "sort": "Newest",
            "image_only": "false",
            "page_locale": "en_US",
            "_noconfig": "true",
            "apikey": API_KEY
        }
        
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code != 200:
                print(f"❌ 서버 응답 에러: {response.status_code}")
                break
                
            data = response.json()
            results = data.get('results', [])
            
            if not results or not results[0].get('reviews'):
                print(f"✅ 데이터 종점에 도달했습니다.")
                break
            
            reviews = results[0].get('reviews', [])
            for rev in reviews:
                details = rev.get("details", {})
                metrics = rev.get("metrics", {})
                
                # 날짜 변환
                raw_ms_date = details.get("created_date")
                formatted_date = datetime.fromtimestamp(raw_ms_date / 1000.0).strftime('%Y-%m-%d %H:%M') if raw_ms_date else "N/A"
                
                raw_reviews.append({
                    "product_id": product_id,
                    "review_id": rev.get("review_id"),
                    "author": details.get("nickname"),
                    "rating": metrics.get("rating"),
                    "headline": details.get("headline"),
                    "comment": details.get("comments"),
                    "date": formatted_date,
                    "raw_timestamp": raw_ms_date 
                })
            
            print(f"🔄 수집 중... (누적: {len(raw_reviews)}개)", end='\r')
            paging_from += page_size
            time.sleep(random.uniform(0.4, 0.7))

        except Exception as e:
            print(f"\n❌ 리뷰 수집 중 중단됨: {e}")
            break

    # 최신순 정렬 및 idx 부여
    raw_reviews.sort(key=lambda x: x['raw_timestamp'], reverse=True)

    final_indexed_reviews = []
    for i, r in enumerate(raw_reviews, 1):
        refined_item = {"idx": i}
        refined_item.update(r)
        del refined_item["raw_timestamp"]
        final_indexed_reviews.append(refined_item)

    print(f"\n✨ {product_name} 리뷰 총 {len(final_indexed_reviews)}개 정제 완료.")
    return final_indexed_reviews

def fetch_ulta_data():
    options = uc.ChromeOptions()
    options.add_argument('--window-size=1920,1080')
    driver = uc.Chrome(options=options)
    
    target_url = "https://www.ulta.com/shop/skin-care/all?sort=best_sellers"
    
    try:
        print(f"📡 Ulta 베스트셀러 페이지 접속 중...")
        driver.get(target_url)
        time.sleep(random.uniform(7, 10))

        soup = BeautifulSoup(driver.page_source, "html.parser")
        items = soup.select('li.ProductListingResults__productCard')
        
        rank_data_list = []
        review_master_list = [] # JSONL 저장을 위해 리스트로 관리
        rank_count = 1
        
        for item in items:
            if rank_count > 10: break
            if item.select_one(".pal-c-ProductCardFooter__sponsored"): continue

            try:
                brand = item.select_one(".pal-c-ProductCardBody--brandName p").get_text(strip=True)
                title = item.select_one("a.pal-c-Link").get_text(strip=True)
                product_url = item.select_one("a.pal-c-Link").get('href')
                if not product_url.startswith('http'):
                    product_url = "https://www.ulta.com" + product_url

                product_id = product_url.split('-')[-1].split('?')[0]

                price_tag = item.select_one(".pal-c-Price span[aria-hidden='true']") or \
                            item.select_one(".pal-c-Price__priceContainer span")
                price = price_tag.get_text(strip=True) if price_tag else "N/A"

                rating_info = item.select_one(".pal-c-Ratings span.sr-only")
                rating = 0.0
                reviews_count = 0
                if rating_info:
                    raw_text = rating_info.get_text(strip=True)
                    rating = float(raw_text.split(' ')[0])
                    if ';' in raw_text:
                        reviews_count = int(raw_text.split(';')[-1].replace('reviews', '').replace(',', '').strip())

                product_json = {
                    "rank": rank_count,
                    "brand": brand,
                    "title": title,
                    "rating": rating,
                    "reviews": reviews_count,
                    "price": price,
                    "url": product_url,
                    "product_id": product_id,
                    "platform": "Ulta",
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                }
                rank_data_list.append(product_json)
                print(f"📍 {rank_count}위 확인: {brand}")

                # 2위 상품만 상세 리뷰 수집
                if rank_count == 2:
                    reviews = get_product_reviews_refined(product_id, title)
                    review_master_list.extend(reviews) # 개별 리뷰들을 리스트에 추가

                rank_count += 1
                
            except Exception as e:
                print(f"⚠️ {rank_count}위 파싱 오류: {e}")
                continue

        # --- [수정 포인트] JSONL 저장 로직 ---
        
        # 1. 순위 데이터 저장 (한 줄에 상품 1개)
        with open(RANK_SAVE_FILE, 'w', encoding='utf-8') as f:
            for entry in rank_data_list:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")
        
        # 2. 리뷰 데이터 저장 (한 줄에 리뷰 1개)
        with open(REVIEW_SAVE_FILE, 'w', encoding='utf-8') as f:
            for review in review_master_list:
                f.write(json.dumps(review, ensure_ascii=False) + "\n")

        print(f"\n📊 JSONL 저장 완료")
        print(f"- 순위 파일: {RANK_SAVE_FILE}")
        print(f"- 리뷰 파일: {REVIEW_SAVE_FILE} (총 {len(review_master_list)}행)")

    finally:
        driver.quit()

if __name__ == "__main__":
    fetch_ulta_data()

## ulta top10 순위 및 모든리뷰데이터 추출코드

In [1]:
import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import requests
import json
import time
import random
from datetime import datetime

# ── 설정 ──────────────────────────────────────────────────────────
API_KEY          = "daa0f241-c242-4483-afb7-4449942d1a2b"
REVIEW_SAVE_FILE = "ulta_reviews_master.jsonl"
RANK_SAVE_FILE   = "ulta_rankings_current.jsonl"


# ── 함수: 단일 상품 리뷰 전체 수집 ───────────────────────────────
def get_product_reviews_refined(product_id, product_name):
    raw_reviews = []
    page_size   = 25
    paging_from = 0

    print(f"\n  🚀 [{product_name[:30]}] 리뷰 수집 시작...")

    while True:
        url = f"https://display.powerreviews.com/m/6406/l/en_US/product/{product_id}/reviews"
        params = {
            "paging.from" : paging_from,
            "paging.size" : page_size,
            "sort"        : "Newest",
            "image_only"  : "false",
            "page_locale" : "en_US",
            "_noconfig"   : "true",
            "apikey"      : API_KEY,
        }
        try:
            response = requests.get(url, params=params, timeout=10)
            if response.status_code != 200:
                print(f"  ❌ 서버 응답 에러: {response.status_code}")
                break

            data    = response.json()
            results = data.get("results", [])

            if not results or not results[0].get("reviews"):
                print(f"  ✅ 데이터 종점 도달 (총 {len(raw_reviews)}개)")
                break

            for rev in results[0].get("reviews", []):
                details   = rev.get("details", {})
                metrics   = rev.get("metrics", {})
                raw_ms    = details.get("created_date")
                fmt_date  = datetime.fromtimestamp(raw_ms / 1000.0).strftime("%Y-%m-%d %H:%M") if raw_ms else "N/A"

                raw_reviews.append({
                    "product_id"   : product_id,
                    "review_id"    : rev.get("review_id"),
                    "author"       : details.get("nickname"),
                    "rating"       : metrics.get("rating"),
                    "headline"     : details.get("headline"),
                    "comment"      : details.get("comments"),
                    "date"         : fmt_date,
                    "raw_timestamp": raw_ms,
                })

            print(f"  🔄 수집 중... (누적: {len(raw_reviews)}개)", end="\r")
            paging_from += page_size
            time.sleep(random.uniform(0.4, 0.7))

        except Exception as e:
            print(f"\n  ❌ 리뷰 수집 중 중단됨: {e}")
            break

    # 최신순 정렬 + idx 부여 + raw_timestamp 제거
    raw_reviews.sort(key=lambda x: x["raw_timestamp"] or 0, reverse=True)
    final = []
    for i, r in enumerate(raw_reviews, 1):
        entry = {"idx": i}
        entry.update(r)
        del entry["raw_timestamp"]
        final.append(entry)

    print(f"\n  ✨ {len(final)}개 정제 완료")
    return final


# ── 메인: Ulta Top 10 수집 + 전 상품 리뷰 수집 ───────────────────
def fetch_ulta_data():
    options = uc.ChromeOptions()
    options.add_argument("--window-size=1920,1080")
    driver = uc.Chrome(options=options)

    try:
        print("📡 Ulta 베스트셀러 페이지 접속 중...")
        driver.get("https://www.ulta.com/shop/skin-care/all?sort=best_sellers")
        time.sleep(random.uniform(7, 10))

        soup  = BeautifulSoup(driver.page_source, "html.parser")
        items = soup.select("li.ProductListingResults__productCard")

        rank_data_list    = []
        all_review_master = []   # 10개 상품 리뷰 전부 누적
        rank_count        = 1

        for item in items:
            if rank_count > 10:
                break

            # 광고 제외
            if item.select_one(".pal-c-ProductCardFooter__sponsored"):
                continue

            try:
                brand = item.select_one(".pal-c-ProductCardBody--brandName p").get_text(strip=True)

                title_link  = item.select_one("a.pal-c-Link")
                title       = title_link.get_text(strip=True)
                product_url = title_link.get("href")
                if not product_url.startswith("http"):
                    product_url = "https://www.ulta.com" + product_url

                product_id = product_url.split("-")[-1].split("?")[0]

                price_tag = (
                    item.select_one(".pal-c-Price span[aria-hidden='true']")
                    or item.select_one(".pal-c-Price__priceContainer span")
                )
                price = price_tag.get_text(strip=True) if price_tag else "N/A"

                rating_info  = item.select_one(".pal-c-Ratings span.sr-only")
                rating       = 0.0
                reviews_count = 0
                if rating_info:
                    raw_text = rating_info.get_text(strip=True)
                    rating   = float(raw_text.split(" ")[0]) if raw_text.split(" ")[0].replace(".", "").isdigit() else 0.0
                    if ";" in raw_text:
                        reviews_count = int(raw_text.split(";")[-1].replace("reviews", "").replace(",", "").strip())

                product_json = {
                    "rank"        : rank_count,
                    "brand"       : brand,
                    "title"       : title,
                    "rating"      : rating,
                    "reviews"     : reviews_count,
                    "price"       : price,
                    "url"         : product_url,
                    "product_id"  : product_id,
                    "platform"    : "Ulta",
                    "collected_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                }
                rank_data_list.append(product_json)
                print(f"\n📍 {rank_count}위: [{brand}] {title[:35]}")

                # ── 전 상품 리뷰 수집 ──────────────────────────────
                reviews = get_product_reviews_refined(product_id, title)
                all_review_master.extend(reviews)
                print(f"  → 누적 리뷰: {len(all_review_master)}개")

                rank_count += 1

            except Exception as e:
                print(f"⚠️ {rank_count}위 파싱 오류: {e}")
                continue

        # ── 저장 ────────────────────────────────────────────────────
        with open(RANK_SAVE_FILE, "w", encoding="utf-8") as f:
            for entry in rank_data_list:
                f.write(json.dumps(entry, ensure_ascii=False) + "\n")

        with open(REVIEW_SAVE_FILE, "w", encoding="utf-8") as f:
            for review in all_review_master:
                f.write(json.dumps(review, ensure_ascii=False) + "\n")

        # ── 최종 결과 출력 ───────────────────────────────────────────
        print()
        print("=" * 65)
        print("🏆 Ulta 스킨케어 베스트셀러 Top 10")
        print("=" * 65)
        for item in rank_data_list:
            print(f"{item['rank']:>2}위 | {item['brand']:<20} | {item['title'][:28]:<28} | ⭐{item['rating']} | {item['price']}")
        print("=" * 65)
        print(f"\n📊 JSONL 저장 완료")
        print(f"  - 순위 파일 : {RANK_SAVE_FILE} ({len(rank_data_list)}개 상품)")
        print(f"  - 리뷰 파일 : {REVIEW_SAVE_FILE} (총 {len(all_review_master)}개 리뷰)")

    finally:
        driver.quit()


if __name__ == "__main__":
    fetch_ulta_data()

📡 Ulta 베스트셀러 페이지 접속 중...

📍 1위: [IT Cosmetics] IT Cosmetics Do It All Sheer Tint F

  🚀 [IT Cosmetics Do It All Sheer T] 리뷰 수집 시작...
  ✅ 데이터 종점 도달 (총 3166개))

  ✨ 3166개 정제 완료
  → 누적 리뷰: 3166개

📍 2위: [medicube] medicube Zero Pore Pad

  🚀 [medicube Zero Pore Pad] 리뷰 수집 시작...
  ✅ 데이터 종점 도달 (총 122개))

  ✨ 122개 정제 완료
  → 누적 리뷰: 3288개

📍 3위: [The Ordinary] The Ordinary Glycolic Acid 7% Exfol

  🚀 [The Ordinary Glycolic Acid 7% ] 리뷰 수집 시작...
  ✅ 데이터 종점 도달 (총 2068개))

  ✨ 2068개 정제 완료
  → 누적 리뷰: 5356개

📍 4위: [Clinique] Clinique Moisture Surge 100H Auto-R

  🚀 [Clinique Moisture Surge 100H A] 리뷰 수집 시작...
  ✅ 데이터 종점 도달 (총 4188개))

  ✨ 4188개 정제 완료
  → 누적 리뷰: 9544개

📍 5위: [MAËLYS] MAËLYS GET-DREAMY Overnight Toning 

  🚀 [MAËLYS GET-DREAMY Overnight To] 리뷰 수집 시작...
  ✅ 데이터 종점 도달 (총 4987개))

  ✨ 4987개 정제 완료
  → 누적 리뷰: 14531개

📍 6위: [The Ordinary] The Ordinary Hyaluronic Acid 2% + B

  🚀 [The Ordinary Hyaluronic Acid 2] 리뷰 수집 시작...
  ✅ 데이터 종점 도달 (총 1274개))

  ✨ 1274개 정제 완료
  → 누적 리뷰: 15805개

📍 7위: [